# Advanced Bedrock Client Examples

This notebook demonstrates all 4 Bedrock APIs with multi-turn conversations and advanced thinking:
- **Converse**: Multi-turn conversations with context retention
- **Converse Stream**: Real-time streaming responses
- **Invoke Model**: Direct model invocation with custom payloads
- **Invoke Model Stream**: Streaming model responses

## Features Covered
- 💬 Multi-turn chat conversations with followup questions
- 🖼️ Multi-modal interactions (text + images)
- 🧠 Advanced thinking and problem-solving with visible thinking responses
- 🔧 Tool use with extended thinking
- ⚙️ Dynamic system prompt modification
- 🔄 Context retention across conversation turns


# Initialization and Configuration


In [ ]:
import json
import base64
import io
from typing import Dict, Any
from datetime import datetime

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from PIL import Image
import numpy as np

# Configuration
MODELS = {
    "chat": "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "thinking": "us.anthropic.claude-sonnet-4-20250514-v1:0",
    "multimodal": "us.anthropic.claude-sonnet-4-20250514-v1:0"
}
AWS_REGION = "us-east-1"
API_URL = "https://your-gateway-alb-url.region.elb.amazonaws.com"
SECRET_ID = "bedrock-gateway-dev-oauth-credentials"

# Extract the secret value from Secret Manager
JSON_SECRET = json.loads(boto3.client('secretsmanager', region_name=AWS_REGION).get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

## Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

## Bedrock Runtime Client Setup

In [ ]:
# Generate token and create client
TOKEN = generate_token()
bedrock = create_bedrock_client(TOKEN)
print(f"✅ Token generated: {TOKEN[:20]}...")
print("✅ Bedrock client configured successfully")

## Helper Functions

In [ ]:

def create_sample_image() -> str:
    """Create a sample image and return base64 encoded string"""
    img = Image.new('RGB', (100, 100), color='lightblue')
    pixels = np.array(img)
    center_x, center_y = 50, 50
    y, x = np.ogrid[:100, :100]
    mask = (x - center_x)**2 + (y - center_y)**2 <= 25**2
    pixels[mask] = [255, 100, 100]  # Red circle
    img = Image.fromarray(pixels)
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def create_sample_image() -> str:
    """Create a sample image and return base64 encoded string"""
    img = Image.new('RGB', (100, 100), color='lightblue')
    pixels = np.array(img)
    center_x, center_y = 50, 50
    y, x = np.ogrid[:100, :100]
    mask = (x - center_x)**2 + (y - center_y)**2 <= 25**2
    pixels[mask] = [255, 100, 100]  # Red circle

    img = Image.fromarray(pixels)
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def create_sample_image_bytes() -> bytes:
    """Create a sample image and return as bytes"""
    img = Image.new('RGB', (100, 100), color='lightblue')
    pixels = np.array(img)
    center_x, center_y = 50, 50
    y, x = np.ogrid[:100, :100]
    mask = (x - center_x)**2 + (y - center_y)**2 <= 25**2
    pixels[mask] = [255, 100, 100]  # Red circle

    img = Image.fromarray(pixels)
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    return buffer.getvalue()

def get_base64_encoded_image(image_path):
    with open(image_path, "rb") as image_file:
        binary_data = image_file.read()
        base_64_encoded_data = base64.b64encode(binary_data)
        base64_string = base_64_encoded_data.decode('utf-8')
        return base64_string

def mock_weather_api(location: str) -> str:
    """Mock weather API for demonstration"""
    weather_data = {
        "paris": "Partly cloudy, 18°C (64°F), light breeze from the west",
        "london": "Overcast, 15°C (59°F), moderate rain expected",
        "tokyo": "Sunny, 22°C (72°F), clear skies",
        "new york": "Cloudy, 20°C (68°F), chance of showers"
    }
    return weather_data.get(location.lower(), f"Weather data not available for {location}")

def print_response(response: Dict[str, Any], title: str = "Response"):
    """Pretty print response with timestamp and thinking content"""
    print(f"\n{'='*50}")
    print(f"{title} - {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*50}")
    if 'output' in response and 'message' in response['output']:
        message = response['output']['message']
        # Display thinking content if available
        if 'content' in message:
            for content_block in message['content']:
                if content_block.get('type') == 'thinking':
                    print("🧠 THINKING:")
                    print("-" * 30)
                    print(content_block.get('thinking', content_block.get('text', '')))
                    print("-" * 30)
                    print()
                elif content_block.get('type') == 'tool_use':
                    print(f"🔧 TOOL USE: {content_block['name']}")
                    print(f"Input: {json.dumps(content_block['input'], indent=2)}")
                    print()
                elif content_block.get('type') == 'text' or 'text' in content_block:
                    print("💬 RESPONSE:")
                    print(content_block.get('text', content_block))
    else:
        print(json.dumps(response, indent=2))
    print(f"{'='*50}\n")

# Scenario 1: Multi-turn Chat Conversations with All 4 APIs

Testing conversational AI with context retention and followup questions.


### Converse

In [ ]:
# Multi-turn conversation using Converse API
print("🔄 Testing Multi-turn Chat with Converse API")

# Initialize conversation with first question
conversation = [
    {"role": "user", "content": [{"text": "I'm planning a trip to Japan. What are the must-visit cities?"}]}
]

# First response
response = bedrock.converse(
    modelId=MODELS["chat"],
    messages=conversation,
    inferenceConfig={"maxTokens": 300, "temperature": 0.7}
)

print_response(response, "Japan Travel - Cities")

# Add assistant response to conversation
conversation.append({
    "role": "assistant",
    "content": response['output']['message']['content']
})

# Add followup question
conversation.append({
    "role": "user",
    "content": [{"text": "What's the best time of year to visit these cities? I'm particularly interested in cherry blossom season."}]
})

# Second response with context
response2 = bedrock.converse(
    modelId=MODELS["chat"],
    messages=conversation,
    inferenceConfig={"maxTokens": 300, "temperature": 0.7}
)

print_response(response2, "Japan Travel - Best Time (Followup)")

### Converse Stream

In [ ]:
# Multi-turn conversation using Converse Stream API
print("🔄 Testing Multi-turn Chat with Converse Stream API")

# Continue the conversation with another followup
conversation.append({
    "role": "assistant",
    "content": response2['output']['message']['content']
})

conversation.append({
    "role": "user",
    "content": [{"text": "Based on our discussion, can you create a 7-day itinerary focusing on Tokyo and Kyoto during cherry blossom season?"}]
})

print("\n" + "="*50)
print("Streaming Multi-turn Response - 7-Day Itinerary")
print("="*50)

streaming_response = bedrock.converse_stream(
    modelId=MODELS["chat"],
    messages=conversation,
    inferenceConfig={"maxTokens": 500, "temperature": 0.7}
)

for chunk in streaming_response["stream"]:
    if "contentBlockDelta" in chunk:
        text = chunk["contentBlockDelta"]["delta"]["text"]
        print(text, end="", flush=True)

print("\n" + "="*50 + "\n")

### Invoke Model

In [ ]:
# Multi-turn conversation using Invoke Model API
print("🔄 Testing Multi-turn Chat with Invoke Model API")

# Convert conversation to Anthropic format
anthropic_messages = []
for msg in conversation:
    if msg["role"] == "user":
        anthropic_messages.append({
            "role": "user",
            "content": msg["content"][0]["text"]
        })
    elif msg["role"] == "assistant":
        anthropic_messages.append({
            "role": "assistant",
            "content": msg["content"][0]["text"]
        })

# Add new question about budget
anthropic_messages.append({
    "role": "user",
    "content": "What would be a reasonable budget for this 7-day trip, including accommodation, food, and transportation?"
})

payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 400,
    "temperature": 0.7,
    "messages": anthropic_messages
}

response = bedrock.invoke_model(
    modelId=MODELS["chat"],
    body=json.dumps(payload)
)

result = json.loads(response['body'].read())
print("\n" + "="*50)
print("Multi-turn Budget Discussion - Invoke Model API")
print("="*50)
print(result['content'][0]['text'])
print("="*50 + "\n")

### Invoke Model Stream

In [ ]:
# Multi-turn conversation using Invoke Model Stream API
print("🔄 Testing Multi-turn Chat with Invoke Model Stream API")

# Add final followup question
anthropic_messages.append({
    "role": "assistant",
    "content": result['content'][0]['text']
})

anthropic_messages.append({
    "role": "user",
    "content": "Can you suggest some money-saving tips specifically for this Japan trip that could reduce the budget by 20-30%?"
})

payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 400,
    "temperature": 0.7,
    "messages": anthropic_messages
}

print("\n" + "="*50)
print("Streaming Multi-turn Response - Money-Saving Tips")
print("="*50)

streaming_response = bedrock.invoke_model_with_response_stream(
    modelId=MODELS["chat"],
    body=json.dumps(payload)
)

for event in streaming_response['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if chunk['type'] == 'content_block_delta':
        print(chunk['delta']['text'], end='', flush=True)

print("\n" + "="*50 + "\n")

# Scenario 2: Advanced Thinking Tasks

Testing complex thinking, problem-solving, and analytical capabilities with visible reasoning.

- See [AWS Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/claude-messages-supported-models.html) for latest supported models


### Thinking with Converse

In [ ]:
# Complex thinking with Converse API (without thinking feature)
print("🧠 Testing Advanced Problem-Solving with Converse API")

thinking_conversation = [
    {"role": "user", "content": [{"text": "A company has 3 departments: Sales, Marketing, and Engineering. The total budget is $1M. Sales gets 40% of the budget, Marketing gets 25%, and Engineering gets the rest. If Engineering hires 5 new employees at $80K each, what percentage of their original budget will be left?"}]}
]

response = bedrock.converse(
    modelId=MODELS["thinking"],
    messages=thinking_conversation,
    inferenceConfig={"maxTokens": 400, "temperature": 0.3}
)

print_response(response, "Budget Calculation - Converse API")

# Add followup thinking question
thinking_conversation.append({
    "role": "assistant",
    "content": response['output']['message']['content']
})

thinking_conversation.append({
    "role": "user",
    "content": [{"text": "Now, if the company wants to maintain the same department budget ratios but increase the total budget to accommodate Engineering's new hires without reducing their remaining budget, what should the new total budget be?"}]
})

response2 = bedrock.converse(
    modelId=MODELS["thinking"],
    messages=thinking_conversation,
    inferenceConfig={"maxTokens": 400, "temperature": 0.3}
)

print_response(response2, "Advanced Budget Optimization - Converse API")

### Thinking with Converse Stream

In [ ]:
# Complex thinking with Converse Stream API
print("🧠 Testing Advanced Thinking with Converse Stream API")

print("\n" + "="*50)
print("Streaming Advanced Logic Puzzle with Thinking")
print("="*50)

streaming_response = bedrock.converse_stream(
    modelId=MODELS["thinking"],
    messages=[
        {
            "role": "user",
            "content": [{
                "text": "You have 8 balls that look identical. 7 of them weigh the same, but 1 is heavier. You have a balance scale that you can use only twice. How do you find the heavier ball? Show your thinking process."
            }]
        }
    ],
    inferenceConfig={"maxTokens": 3000, "temperature": 1.0}
)

current_block_type = None
for chunk in streaming_response["stream"]:
    if "contentBlockStart" in chunk:
        current_block_type = chunk["contentBlockStart"]["contentBlockType"]
        if current_block_type == "thinking":
            print("\n🧠 THINKING:")
            print("-" * 30)
        elif current_block_type == "text":
            print("\n💬 RESPONSE:")

    elif "contentBlockDelta" in chunk:
        delta = chunk["contentBlockDelta"]["delta"]
        if "thinking" in delta:
            print(delta["thinking"], end="", flush=True)
        elif "text" in delta:
            print(delta["text"], end="", flush=True)

    elif "contentBlockStop" in chunk and current_block_type == "thinking":
        print("\n" + "-" * 30)

print("\n" + "="*50 + "\n")

### Thinking with Invoke Model

In [ ]:
# Complex thinking with Invoke Model API (with thinking feature)
print("🧠 Testing Advanced Thinking with Invoke Model API")

complex_thinking = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 3000,
    "temperature": 1.0,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2024
    },
    "messages": [
        {
            "role": "user",
            "content": "You have 8 balls that look identical. 7 of them weigh the same, but 1 is heavier. You have a balance scale that you can use only twice. How do you find the heavier ball? Show your thinking process."
        }
    ]
}

response = bedrock.invoke_model(
    modelId=MODELS["thinking"],
    body=json.dumps(complex_thinking)
)

result = json.loads(response['body'].read())
print("\n" + "="*50)
print("Advanced Logic Puzzle with Thinking - Invoke Model API")
print("="*50)

# Display thinking and response content
for content_block in result['content']:
    if content_block.get('type') == 'thinking':
        print("🧠 THINKING:")
        print("-" * 30)
        print(content_block.get('thinking', content_block.get('text', '')))
        print("-" * 30)
        print()
    elif content_block.get('type') == 'text':
        print("💬 RESPONSE:")
        print(content_block['text'])
print("="*50 + "\n")

### Thinking with Invoke Model Stream

In [ ]:
# Complex thinking with Invoke Model Stream API
print("🧠 Testing Advanced Thinking with Invoke Model Stream API")

complex_thinking_stream = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 3000,
    "temperature": 1.0,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2024
    },
    "messages": [
        {
            "role": "user",
            "content": "Solve this step by step: If a train travels 120 miles in 2 hours, and then increases its speed by 50% for the next 3 hours, how far does it travel in total?"
        }
    ]
}

print("\n" + "="*50)
print("Streaming Advanced Math Problem with Thinking")
print("="*50)

streaming_response = bedrock.invoke_model_with_response_stream(
    modelId=MODELS["thinking"],
    body=json.dumps(complex_thinking_stream)
)

current_block_type = None
for event in streaming_response['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if chunk['type'] == 'content_block_start':
        current_block_type = chunk['content_block']['type']
        if current_block_type == 'thinking':
            print("\n🧠 THINKING:")
            print("-" * 30)
        elif current_block_type == 'text':
            print("\n💬 RESPONSE:")
    elif chunk['type'] == 'content_block_delta':
        if 'text' in chunk['delta']:
            print(chunk['delta']['text'], end='', flush=True)
        elif 'thinking' in chunk['delta']:
            print(chunk['delta']['thinking'], end='', flush=True)
    elif chunk['type'] == 'content_block_stop' and current_block_type == 'thinking':
        print("\n" + "-" * 30)

print("\n" + "="*50 + "\n")

## Thinking with Tool

### Thinking with Tool - Converse

In [ ]:
# Thinking with tools using Converse API - Single conversation
print("🧠🔧 Testing Thinking with Tools - Converse API")

response = bedrock.converse(
    modelId=MODELS["thinking"],
    messages=[
        {
            "role": "user",
            "content": [{
                "text": "I'm planning to visit Paris tomorrow. Can you check the weather and suggest what I should wear based on the conditions?"
            }]
        }
    ],
    toolConfig={
        "tools": [
            {
                "toolSpec": {
                    "name": "get_weather",
                    "description": "Get current weather information for a specific location",
                    "inputSchema": {
                        "json": {
                            "type": "object",
                            "properties": {
                                "location": {
                                    "type": "string",
                                    "description": "The city or location to get weather for"
                                }
                            },
                            "required": ["location"]
                        }
                    }
                }
            }
        ]
    },
    inferenceConfig={"maxTokens": 3000, "temperature": 1.0}
)

# Display response and simulate tool execution if tool_use is present
print_response(response, "Weather Check with Tool Use - Converse API")

# If tool was used, show the simulated result
for content in response['output']['message']['content']:
    if content.get('type') == 'tool_use':
        location = content['input'].get('location', 'unknown')
        weather_result = mock_weather_api(location)
        print(f"🔧 Tool Result: {weather_result}")

### Thinking with Tool - Converse Stream

In [ ]:
# Thinking with tools using Converse Stream API
print("🧠🔧 Testing Thinking with Tools - Converse Stream API")

print("\n" + "="*50)
print("Streaming Weather Analysis with Thinking")
print("="*50)

streaming_response = bedrock.converse_stream(
    modelId=MODELS["thinking"],
    messages=[
        {
            "role": "user",
            "content": [{
                "text": "I need to decide what to pack for a business trip to London next week. Check the weather forecast and recommend appropriate clothing."
            }]
        }
    ],
    toolConfig={
        "tools": [
            {
                "toolSpec": {
                    "name": "get_weather",
                    "description": "Get current weather information for a specific location",
                    "inputSchema": {
                        "json": {
                            "type": "object",
                            "properties": {
                                "location": {
                                    "type": "string",
                                    "description": "The city or location to get weather for"
                                }
                            },
                            "required": ["location"]
                        }
                    }
                }
            }
        ]
    },
    inferenceConfig={"maxTokens": 3000, "temperature": 1.0}
)

current_block_type = None
tool_input = ""

for chunk in streaming_response["stream"]:
    if "contentBlockStart" in chunk:
        start_info = chunk["contentBlockStart"]["start"]
        if "toolUse" in start_info:
            current_block_type = "toolUse"
            tool_name = start_info["toolUse"]["name"]
            print(f"\n🔧 TOOL USE: {tool_name}")
        elif "text" in start_info:
            current_block_type = "text"
            print("\n💬 RESPONSE:")

    elif "contentBlockDelta" in chunk:
        delta = chunk["contentBlockDelta"]["delta"]
        if "toolUse" in delta:
            tool_input += delta["toolUse"]["input"]
            print(delta["toolUse"]["input"], end="", flush=True)
        elif "text" in delta:
            print(delta["text"], end="", flush=True)

    elif "contentBlockStop" in chunk:
        if current_block_type == "toolUse":
            try:
                tool_data = json.loads(tool_input)
                location = tool_data.get("location", "unknown")
                weather_result = mock_weather_api(location)
                print(f"\nResult: {weather_result}")
            except json.JSONDecodeError:
                print("\nTool input parsing error")
            tool_input = ""

print("\n" + "="*50 + "\n")

### Thinking with Tool - Invoke Model

In [ ]:
# Thinking with tools using Invoke Model API
print("🧠🔧 Testing Thinking with Tools - Invoke Model API")

payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 3000,
    "temperature": 1.0,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2024
    },
    "tools": [
        {
            "name": "get_weather",
            "description": "Get current weather information for a specific location",
            "input_schema": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city or location to get weather for"
                    }
                },
                "required": ["location"]
            }
        }
    ],
    "messages": [
        {
            "role": "user",
            "content": "I'm organizing an outdoor wedding in Tokyo this weekend. Can you check the weather and advise if we need backup plans?"
        }
    ]
}

response = bedrock.invoke_model(
    modelId=MODELS["thinking"],
    body=json.dumps(payload)
)

result = json.loads(response['body'].read())
print("\n" + "="*50)
print("Wedding Planning with Weather Check - Invoke Model API")
print("="*50)

for content_block in result['content']:
    if content_block.get('type') == 'thinking':
        print("🧠 THINKING:")
        print("-" * 30)
        print(content_block.get('thinking'))
        print("-" * 30)
        print()
    elif content_block.get('type') == 'tool_use':
        print(f"🔧 TOOL USE: {content_block['name']}")
        print(f"Input: {json.dumps(content_block['input'], indent=2)}")
        # Mock tool execution
        location = content_block['input'].get('location', 'unknown')
        weather_result = mock_weather_api(location)
        print(f"Result: {weather_result}")
        print()
    elif content_block.get('type') == 'text':
        print("💬 RESPONSE:")
        print(content_block['text'])

print("="*50 + "\n")

### Thinking with Tool - Invoke Model Stream

In [ ]:
# Tool use with extended thinking (streaming)
print("\n" + "="*60)
print("🔧 Testing Tool Use with Extended Thinking (Streaming)")
print("="*60)

tool_thinking_payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 3000,
    "temperature": 1.0,
    "thinking": {
        "type": "enabled",
        "budget_tokens": 2024
    },
    "tools": [
        {
            "name": "get_weather",
            "description": "Get current weather information for a specific location",
            "input_schema": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city or location to get weather for"
                    }
                },
                "required": ["location"]
            }
        }
    ],
    "messages": [
        {
            "role": "user",
            "content": "I'm planning to visit Paris tomorrow. Can you check the weather and suggest what I should wear?"
        }
    ]
}

print("\nStreaming Tool Use with Thinking - Weather Check")
print("="*50)

streaming_response = bedrock.invoke_model_with_response_stream(
    modelId=MODELS["thinking"],
    body=json.dumps(tool_thinking_payload)
)

current_block_type = None
tool_name = None
tool_input = ""

for event in streaming_response['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if chunk['type'] == 'content_block_start':
        current_block_type = chunk['content_block']['type']
        if current_block_type == 'thinking':
            print("\n🧠 THINKING:")
            print("-" * 30)
        elif current_block_type == 'tool_use':
            tool_name = chunk['content_block']['name']
            print(f"\n🔧 TOOL USE: {tool_name}")
        elif current_block_type == 'text':
            print("\n💬 RESPONSE:")
    elif chunk['type'] == 'content_block_delta':
        if 'thinking' in chunk['delta']:
            print(chunk['delta']['thinking'], end='', flush=True)
        elif 'partial_json' in chunk['delta']:
            tool_input += chunk['delta']['partial_json']
            print(chunk['delta']['partial_json'], end='', flush=True)
        elif 'text' in chunk['delta']:
            print(chunk['delta']['text'], end='', flush=True)
    elif chunk['type'] == 'content_block_stop':
        if current_block_type == 'thinking':
            print("\n" + "-" * 30)
        elif current_block_type == 'tool_use' and tool_name == 'get_weather':
            try:
                tool_data = json.loads(tool_input)
                location = tool_data.get('location', 'unknown')
                weather_result = mock_weather_api(location)
                print(f"\nResult: {weather_result}")
            except json.JSONDecodeError:
                print("\nTool input parsing error")
            tool_input = ""

print("\n" + "="*50 + "\n")

# Scenario 3: Multi-modal Interactions

Testing text + image combination.


In [ ]:
# Create sample image for multi-modal testing
import base64
from IPython.display import Image as IPImage, display
sample_image_b64 = create_sample_image()
sample_image_bytes = create_sample_image_bytes()
print("✅ Sample image created and encoded")

# Display the sample image
display(IPImage(data=base64.b64decode(sample_image_b64)))

### Multi-modal with Converse

In [ ]:
# Multi-modal with Converse API
print("🖼️ Testing Multi-modal with Converse API")

response = bedrock.converse(
    modelId=MODELS["multimodal"],
    messages=[
        {
            'role': 'user',
            'content': [
                {
                    'text': 'Describe this image and identify the geometric shapes present.'
                },
                {
                    'image': {
                        'format': 'png',
                        'source': {
                            'bytes': sample_image_bytes
                        }
                    }
                }
            ]
        }
    ],
    inferenceConfig={"maxTokens": 400, "temperature": 0.6}
)

print_response(response, "Image Analysis - Converse API")

### Multi-modal with Converse Stream

In [ ]:
# Multi-modal with Converse Stream API
print("🖼️ Testing Multi-modal with Converse Stream API")

print("\n" + "="*50)
print("Streaming Image Analysis - Converse Stream API")
print("="*50)

streaming_response = bedrock.converse_stream(
    modelId=MODELS["multimodal"],
    messages=[
        {
            'role': 'user',
            'content': [
                {
                    'text': 'Analyze this image from an artistic perspective. What colors, composition, and visual elements do you observe?'
                },
                {
                    'image': {
                        'format': 'png',
                        'source': {
                            'bytes': sample_image_bytes
                        }
                    }
                }
            ]
        }
    ],
    inferenceConfig={"maxTokens": 400, "temperature": 0.6}
)

for chunk in streaming_response["stream"]:
    if "contentBlockDelta" in chunk:
        text = chunk["contentBlockDelta"]["delta"]["text"]
        print(text, end="", flush=True)

print("\n" + "="*50 + "\n")

### Multi-modal with Invoke Model

In [ ]:
# Multi-modal with Invoke Model API
print("🖼️ Testing Multi-modal with Invoke Model API")

payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 400,
    "temperature": 0.6,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Analyze this image from a mathematical perspective. What geometric principles are demonstrated?"},
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": sample_image_b64
                    }
                }
            ]
        }
    ]
}

response = bedrock.invoke_model(
    modelId=MODELS["multimodal"],
    body=json.dumps(payload)
)

result = json.loads(response['body'].read())
print("\n" + "="*50)
print("Mathematical Analysis - Invoke Model API")
print("="*50)
print(result['content'][0]['text'])
print("="*50 + "\n")

### Multi-modal with Invoke Stream

In [ ]:
# Multi-modal with Invoke Model Stream API
print("🖼️ Testing Multi-modal with Invoke Model Stream API")

payload = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 400,
    "temperature": 0.6,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image as if you were explaining it to a child. What do you see and what might it represent?"},
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": sample_image_b64
                    }
                }
            ]
        }
    ]
}

print("\n" + "="*50)
print("Streaming Child-Friendly Description - Invoke Model Stream API")
print("="*50)

streaming_response = bedrock.invoke_model_with_response_stream(
    modelId=MODELS["multimodal"],
    body=json.dumps(payload)
)

for event in streaming_response['body']:
    chunk = json.loads(event['chunk']['bytes'])
    if chunk['type'] == 'content_block_delta':
        print(chunk['delta']['text'], end='', flush=True)

print("\n" + "="*50 + "\n")

# Scenario 4: System Prompts and Persona Testing

Testing different system prompts and personas across APIs.


In [ ]:
# System prompts comparison
print("⚙️ Testing System Prompts")

system_prompts = {
    "technical_expert": "You are a senior software engineer with 15 years of experience. Explain concepts clearly with practical examples.",
    "creative_writer": "You are a creative writer who uses vivid metaphors and storytelling to explain complex topics."
}

user_question = "Explain how machine learning works."

# Test with Converse API
for persona, system_prompt in system_prompts.items():
    response = bedrock.converse(
        modelId=MODELS["thinking"],
        messages=[{"role": "user", "content": [{"text": user_question}]}],
        system=[{"text": system_prompt}],
        inferenceConfig={"maxTokens": 250, "temperature": 0.8}
    )
    print_response(response, f"Converse API - {persona.replace('_', ' ').title()}")

# Test with Invoke Model API (with thinking)
for persona, system_prompt in system_prompts.items():
    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 3000,
        "temperature": 1.0,
        "thinking": {
            "type": "enabled",
            "budget_tokens": 2024
        },
        "system": system_prompt,
        "messages": [
            {"role": "user", "content": user_question}
        ]
    }
    response = bedrock.invoke_model(
        modelId=MODELS["thinking"],
        body=json.dumps(payload)
    )
    result = json.loads(response['body'].read())
    print("\n" + "="*50)
    print(f"Invoke Model API with Thinking - {persona.replace('_', ' ').title()}")
    print("="*50)
    # Display thinking and response content
    for content_block in result['content']:
        if content_block.get('type') == 'thinking':
            print("🧠 THINKING:")
            print("-" * 30)
            print(content_block.get('thinking', content_block.get('text', '')))
            print("-" * 30)
            print()
        elif content_block.get('type') == 'text':
            print("💬 RESPONSE:")
            print(content_block['text'])
    print("="*50 + "\n")

# Scenario 5: Performance and Configuration Testing

Testing different configurations across all APIs.


In [ ]:
# Performance testing
print("⚡ Testing Performance Variations Across Parameters")

test_prompt = "Write a haiku about artificial intelligence."
temperatures = [0.1, 0.9]

for temp in temperatures:
    # Converse API
    response = bedrock.converse(
        modelId=MODELS["chat"],
        messages=[{"role": "user", "content": [{"text": test_prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": temp}
    )
    print_response(response, f"Converse API - Temperature {temp}")
    # Invoke Model API
    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 100,
        "temperature": temp,
        "messages": [{"role": "user", "content": test_prompt}]
    }
    response = bedrock.invoke_model(
        modelId=MODELS["chat"],
        body=json.dumps(payload)
    )
    result = json.loads(response['body'].read())
    print("\n" + "="*50)
    print(f"Invoke Model API - Temperature {temp}")
    print("="*50)
    print(result['content'][0]['text'])
    print("="*50 + "\n")